# Formativa 2: Modelamiento predictivo de rotación de personal

**Asignatura:** Estadística Computacional  
**Integrantes:** Hernán Saavedra, Lenin Morales y Rodrigo Gómez  
**Dataset:** IBM HR Employee Attrition

## Objetivo

Preparar los datos para construir un modelo de regresión logística que
estime la probabilidad de salida de un empleado, integrando los hallazgos
de las Sumativas 1 y 2.

Siguiendo las indicaciones del equipo docente, se generarán faltantes
artificiales MCAR en el 10 % de los registros de ingreso mensual, con
semilla fija y respaldo de los valores originales. Esta versión se
conservará para la Sumativa 3.

La preparación incluirá una partición 70/30 estratificada por Attrition,
imputación por mediana y estandarización de los predictores numéricos.
Los parámetros de imputación y estandarización se calcularán
exclusivamente con los datos de entrenamiento.

Los datos son sintéticos; las conclusiones tienen una finalidad
académica y no justifican directamente políticas empresariales.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEMILLA = 42

# Localizar la raíz del repositorio desde la ubicación del notebook.
RAIZ = next(
    (
        carpeta
        for carpeta in [Path.cwd(), *Path.cwd().parents]
        if (carpeta / "datos" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").is_file()
    ),
    None,
)

if RAIZ is None:
    raise FileNotFoundError("No se encontró el dataset original en la carpeta datos.")

DIRECTORIO_F2 = RAIZ / "Formativa2"

for nombre in ["datos", "resultados", "figuras"]:
    (DIRECTORIO_F2 / nombre).mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Raíz del proyecto:", RAIZ)
print("Entorno preparado correctamente.")

Python: 3.14.2
Raíz del proyecto: /workspaces/estadisticaComputacional
Entorno preparado correctamente.


## 1. Carga y verificación del dataset original

Se utiliza el mismo conjunto de datos de S1 y S2. Se comprueban sus
dimensiones, valores faltantes, duplicados y distribución de Attrition
antes de generar los faltantes artificiales. El archivo original se
conserva sin modificaciones.

In [2]:
RUTA_ORIGINAL = (
    RAIZ / "datos" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

df_original = pd.read_csv(RUTA_ORIGINAL)

print("Dimensiones:", df_original.shape)
print("Valores faltantes:", df_original.isna().sum().sum())
print("Filas duplicadas:", df_original.duplicated().sum())

resumen_objetivo = (
    df_original["Attrition"]
    .value_counts()
    .rename_axis("Attrition")
    .to_frame("Cantidad")
)

resumen_objetivo["Porcentaje"] = (
    resumen_objetivo["Cantidad"] / len(df_original) * 100
)

display(resumen_objetivo.round(2))

# Verificar las condiciones esperadas antes de introducir faltantes.
assert df_original.shape == (1470, 35), "Revisar la versión del dataset."
assert df_original.isna().sum().sum() == 0, "El original contiene faltantes."
assert df_original["EmployeeNumber"].is_unique, "Hay identificadores repetidos."
assert set(df_original["Attrition"].unique()) == {"No", "Yes"}

Dimensiones: (1470, 35)
Valores faltantes: 0
Filas duplicadas: 0


,Cantidad,Porcentaje
Attrition,,
No,1233,83.88
Yes,237,16.12


## 2. Generación reproducible de faltantes MCAR

Se seleccionan aleatoriamente, sin reemplazo y con semilla 42, 147
registros (10 %) de MonthlyIncome para ocultar sus valores. Todos los
registros tienen la misma probabilidad de selección, independientemente
del ingreso, de Attrition y de las demás variables.

Se guarda la versión con faltantes y un respaldo de los valores
ocultados, identificados mediante EmployeeNumber. El respaldo se
reservará para evaluar el error de imputación en S3 y no se utilizará
para ajustar los modelos. Attrition se conserva sin modificaciones.

In [3]:
RUTA_MCAR = DIRECTORIO_F2 / "datos" / "IBM_HR_MCAR.csv"
RUTA_RESPALDO = DIRECTORIO_F2 / "datos" / "valores_ocultados_ingreso.csv"

if RUTA_MCAR.exists() and RUTA_RESPALDO.exists():
    df_mcar = pd.read_csv(RUTA_MCAR)
    respaldo_ocultados = pd.read_csv(RUTA_RESPALDO)
    print("Se cargó la versión MCAR previamente guardada.")

elif RUTA_MCAR.exists() or RUTA_RESPALDO.exists():
    raise FileExistsError(
        "Falta uno de los dos archivos MCAR. Revisar antes de continuar."
    )

else:
    rng = np.random.default_rng(SEMILLA)
    cantidad = int(len(df_original) * 0.10)

    indices_ocultados = np.sort(
        rng.choice(df_original.index, size=cantidad, replace=False)
    )

    respaldo_ocultados = df_original.loc[
        indices_ocultados, ["EmployeeNumber", "MonthlyIncome"]
    ].copy()

    df_mcar = df_original.copy()
    df_mcar["MonthlyIncome"] = df_mcar["MonthlyIncome"].astype(float)
    df_mcar.loc[indices_ocultados, "MonthlyIncome"] = np.nan

    respaldo_ocultados.to_csv(RUTA_RESPALDO, index=False)
    df_mcar.to_csv(RUTA_MCAR, index=False)

    print("Se creó y guardó la versión MCAR.")

assert df_mcar.shape == df_original.shape
assert df_mcar["MonthlyIncome"].isna().sum() == 147
assert df_mcar.isna().sum().sum() == 147
assert len(respaldo_ocultados) == 147

pd.testing.assert_frame_equal(
    df_mcar.drop(columns="MonthlyIncome"),
    df_original.drop(columns="MonthlyIncome"),
)

ids_faltantes = set(
    df_mcar.loc[df_mcar["MonthlyIncome"].isna(), "EmployeeNumber"]
)
assert ids_faltantes == set(respaldo_ocultados["EmployeeNumber"])

print("Faltantes en MonthlyIncome:", df_mcar["MonthlyIncome"].isna().sum())
print("Porcentaje:", df_mcar["MonthlyIncome"].isna().mean() * 100)
print("Dataset guardado en:", RUTA_MCAR)
print("Respaldo guardado en:", RUTA_RESPALDO)

Se cargó la versión MCAR previamente guardada.
Faltantes en MonthlyIncome: 147
Porcentaje: 10.0
Dataset guardado en: /workspaces/estadisticaComputacional/Formativa2/datos/IBM_HR_MCAR.csv
Respaldo guardado en: /workspaces/estadisticaComputacional/Formativa2/datos/valores_ocultados_ingreso.csv


## 3. División en entrenamiento y prueba

Se asigna el 70 % de los registros al entrenamiento y el 30 % a prueba,
con semilla 42 y estratificación por Attrition para mantener proporciones
similares de ambas clases.

La partición se realiza antes de imputar y estandarizar. Los parámetros
de esas transformaciones se estimarán únicamente con entrenamiento.
EmployeeNumber se utiliza para identificar los registros y conservar
la partición; no se utilizará como predictor.

In [4]:
RUTA_PARTICION = DIRECTORIO_F2 / "datos" / "particion_70_30.csv"

if RUTA_PARTICION.exists():
    particion = pd.read_csv(RUTA_PARTICION)
    print("Se cargó la partición previamente guardada.")

else:
    ids_train, ids_test = train_test_split(
        df_mcar["EmployeeNumber"].to_numpy(),
        test_size=0.30,
        random_state=SEMILLA,
        stratify=df_mcar["Attrition"].to_numpy(),
    )

    particion = pd.concat(
        [
            pd.DataFrame({
                "EmployeeNumber": ids_train,
                "Conjunto": "Entrenamiento",
            }),
            pd.DataFrame({
                "EmployeeNumber": ids_test,
                "Conjunto": "Prueba",
            }),
        ],
        ignore_index=True,
    )

    particion.to_csv(RUTA_PARTICION, index=False)
    print("Se creó y guardó la partición.")

assert particion["EmployeeNumber"].is_unique
assert set(particion["EmployeeNumber"]) == set(df_mcar["EmployeeNumber"])
assert set(particion["Conjunto"]) == {"Entrenamiento", "Prueba"}

datos_por_id = df_mcar.set_index("EmployeeNumber")

train = datos_por_id.loc[
    particion.loc[
        particion["Conjunto"] == "Entrenamiento", "EmployeeNumber"
    ]
].copy()

test = datos_por_id.loc[
    particion.loc[
        particion["Conjunto"] == "Prueba", "EmployeeNumber"
    ]
].copy()

assert len(train) == 1029
assert len(test) == 441
assert set(train.index).isdisjoint(test.index)

resumen_particion = pd.DataFrame({
    "Registros": [len(train), len(test)],
    "Salidas": [
        train["Attrition"].eq("Yes").sum(),
        test["Attrition"].eq("Yes").sum(),
    ],
    "Porcentaje de salida": [
        train["Attrition"].eq("Yes").mean() * 100,
        test["Attrition"].eq("Yes").mean() * 100,
    ],
}, index=["Entrenamiento", "Prueba"])

display(resumen_particion.round(2))

Se cargó la partición previamente guardada.


,Registros,Salidas,Porcentaje de salida
Entrenamiento,1029,166,16.13
Prueba,441,71,16.10


## 4. Imputación por mediana

El ingreso mensual presentó asimetría positiva en S1. En S2, el análisis
de sensibilidad mostró que los ingresos extremos afectaban la magnitud
de las diferencias entre grupos. Por ello, se utiliza la mediana como
estrategia inicial de imputación.

La mediana se calcula exclusivamente con los ingresos observados del
entrenamiento y se aplica a los faltantes de ambos conjuntos. Los
valores originales ocultados no intervienen en este procedimiento.

Esta estrategia conserva todos los registros, aunque concentra valores
en la mediana y puede alterar la variabilidad y las relaciones entre
variables. En S3 se comparará con otras estrategias de tratamiento.

In [5]:
imputador = SimpleImputer(strategy="median")

train_imputado = train.copy()
test_imputado = test.copy()

# Ajustar exclusivamente con entrenamiento.
train_imputado["MonthlyIncome"] = imputador.fit_transform(
    train[["MonthlyIncome"]]
).ravel()

# Aplicar a prueba la misma mediana, sin recalcularla.
test_imputado["MonthlyIncome"] = imputador.transform(
    test[["MonthlyIncome"]]
).ravel()

mediana_entrenamiento = float(imputador.statistics_[0])

resumen_imputacion = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Ingresos imputados": [
        train["MonthlyIncome"].isna().sum(),
        test["MonthlyIncome"].isna().sum(),
    ],
    "Mediana utilizada": [
        mediana_entrenamiento,
        mediana_entrenamiento,
    ],
})

assert train_imputado.isna().sum().sum() == 0
assert test_imputado.isna().sum().sum() == 0

# Comprobar que los ingresos observados se conservaron.
for antes, despues in [
    (train, train_imputado),
    (test, test_imputado),
]:
    observados = antes["MonthlyIncome"].notna()
    pd.testing.assert_series_equal(
        antes.loc[observados, "MonthlyIncome"],
        despues.loc[observados, "MonthlyIncome"],
    )

resumen_imputacion.to_csv(
    DIRECTORIO_F2 / "resultados" / "resumen_imputacion.csv",
    index=False,
)

display(resumen_imputacion)
print("Imputación completada sin utilizar información de prueba.")

,Conjunto,Ingresos imputados,Mediana utilizada
0,Entrenamiento,96,5067.0
1,Prueba,51,5067.0


Imputación completada sin utilizar información de prueba.


## 5. Predictores seleccionados y relación con S1/S2

| Predictor | Fundamento |
|-----------|------------|
| OverTime | En S1, la salida fue de 30,53 % con horas extra frente a 10,44 % sin ellas. Se observó asociación mediante chi-cuadrado. |
| MonthlyIncome | En S1, el ingreso medio fue mayor entre quienes permanecieron. En S2, los intervalos bootstrap de las diferencias de medias y medianas conservaron una dirección positiva. |
| YearsAtCompany | En S1, la antigüedad media fue de 7,37 años entre quienes permanecieron y 5,13 entre quienes salieron. |
| JobSatisfaction | En S1, la salida fue de 22,84 % en el nivel de satisfacción 1 y de 11,33 % en el nivel 4. |

En S2 se validaron correlaciones positivas entre ingreso y experiencia,
edad y experiencia, y antigüedad y experiencia. Estas relaciones son
entre predictores y no demuestran por sí mismas asociación con Attrition.
Para limitar redundancias, este modelo inicial no incorpora edad ni
experiencia laboral total.

También se excluye JobLevel por la alta correlación con MonthlyIncome
advertida en la retroalimentación docente. La multicolinealidad de los
predictores seleccionados se evaluará mediante VIF.

Se conservan los ingresos extremos: S2 mostró sensibilidad de la
magnitud de las diferencias, pero no justificó considerarlos errores.

JobSatisfaction se tratará como una puntuación ordinal de 1 a 4,
suponiendo un efecto lineal sobre el logit. Esta simplificación se
revisará en S3 mediante una posible codificación categórica.

La selección utiliza análisis exploratorios previos del conjunto
completo; por ello, la evaluación posterior debe considerarse una
validación interna exploratoria.

## 6. Codificación y estandarización

Attrition se codifica como 1 = sale y 0 = permanece. OverTime se codifica
como 1 = sí y 0 = no, conservando su interpretación binaria.

Los otros tres predictores se estandarizan con parámetros calculados
exclusivamente en entrenamiento. Sus odds ratios se interpretarán por
un incremento de una desviación estándar de entrenamiento, manteniendo
constantes los demás predictores.

In [6]:
PREDICTORES = [
    "OverTime",
    "MonthlyIncome",
    "YearsAtCompany",
    "JobSatisfaction",
]

VARIABLES_ESCALAR = [
    "MonthlyIncome",
    "YearsAtCompany",
    "JobSatisfaction",
]

X_train = train_imputado[PREDICTORES].copy()
X_test = test_imputado[PREDICTORES].copy()

for datos in [X_train, X_test]:
    datos["OverTime"] = datos["OverTime"].map({"No": 0, "Yes": 1})

y_train = train_imputado["Attrition"].map({"No": 0, "Yes": 1})
y_test = test_imputado["Attrition"].map({"No": 0, "Yes": 1})

escalador = StandardScaler()

X_train[VARIABLES_ESCALAR] = escalador.fit_transform(
    X_train[VARIABLES_ESCALAR]
)

X_test[VARIABLES_ESCALAR] = escalador.transform(
    X_test[VARIABLES_ESCALAR]
)

parametros_escalado = pd.DataFrame({
    "Variable": VARIABLES_ESCALAR,
    "Media entrenamiento": escalador.mean_,
    "Desviación entrenamiento": escalador.scale_,
})

assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert y_train.notna().all() and y_test.notna().all()
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

parametros_escalado.to_csv(
    DIRECTORIO_F2 / "resultados" / "parametros_escalado.csv",
    index=False,
)

display(parametros_escalado.round(4))
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Predictores preparados correctamente.")

,Variable,Media entrenamiento,Desviación entrenamiento
0,MonthlyIncome,6425.3275,4403.5548
1,YearsAtCompany,6.9388,5.9480
2,JobSatisfaction,2.7065,1.1089


Entrenamiento: (1029, 4)
Prueba: (441, 4)
Predictores preparados correctamente.


## 7. Exportación de insumos para el modelamiento

Se exportan los predictores preparados y la variable objetivo de cada
conjunto, conservando EmployeeNumber como identificador para comprobar
la correspondencia entre filas. Este identificador no es un predictor.

Estos archivos permitirán compartir una preparación común. El notebook
conserva el procedimiento necesario para regenerarlos.

In [8]:
import json

carpeta_datos = DIRECTORIO_F2 / "datos"

for nombre, objeto in [
    ("X_train.csv", X_train),
    ("X_test.csv", X_test),
    ("y_train.csv", y_train.rename("Attrition")),
    ("y_test.csv", y_test.rename("Attrition")),
]:
    objeto.to_csv(
        carpeta_datos / nombre,
        index=True,
        index_label="EmployeeNumber",
    )

configuracion = {
    "semilla": SEMILLA,
    "objetivo": "Attrition",
    "clase_positiva": "Yes = 1: empleado que sale",
    "predictores": PREDICTORES,
    "variables_estandarizadas": VARIABLES_ESCALAR,
    "codificacion_OverTime": {"No": 0, "Yes": 1},
    "mediana_ingreso_entrenamiento": mediana_entrenamiento,
    "n_entrenamiento": len(X_train),
    "n_prueba": len(X_test),
    "umbral_inicial": 0.5,
    "ajuste_imputacion_y_escalado": "Solo entrenamiento",
}

ruta_configuracion = (
    DIRECTORIO_F2 / "resultados" / "configuracion_preparacion.json"
)

ruta_configuracion.write_text(
    json.dumps(configuracion, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Se guardaron X_train, X_test, y_train e y_test.")
print("Se guardó la configuración de preparación.")

Se guardaron X_train, X_test, y_train e y_test.
Se guardó la configuración de preparación.
